# Software FJ · Fase 4

**Programación 213023 · UNAD · Ingeniería de Sistemas**

**Responsable:** Ariel Sandoval Florez · **GitHub:** arielsanflo1
**Grupo:** 213023_109 · **Tutor:** Jaider Lineros

Este notebook demuestra abstracción, encapsulación, herencia, polimorfismo y
manejo de excepciones. Los datos son ficticios y se mantienen en listas; solo
los eventos y errores se escriben en archivos. No hay base de datos.

**Instrucción de ejecución:** descargar el repositorio completo, abrir este
notebook desde su raíz y ejecutar todas las celdas con un kernel limpio.
Se requiere Python 3.10 o posterior.

## 1. Diseño

`Entidad` es una clase abstracta que exige identidad y descripción. `Cliente`
encapsula nombre y contacto. `Servicio` es abstracta y sus derivados son
`Sala`, `Equipo` y `Asesoria`. Cada uno sobrescribe la validación de duración,
la descripción y el cálculo. `Reserva` controla estados y `Sistema` coordina
las listas internas. El código completo está en `software_fj/modelo.py`.

**Regla de disponibilidad:** un servicio admite una reserva confirmada activa.
Procesar o cancelar libera ocupación. Las solicitudes pendientes no bloquean
el recurso. Esta simulación no implementa calendario, pagos ni persistencia.

In [1]:
# Se comprueba la ubicación antes de importar los módulos del repositorio.
from pathlib import Path
assert (Path.cwd() / 'software_fj').is_dir(), 'Abra el notebook desde la raíz del repositorio.'
# Se importan las tres especializaciones bajo el mismo contrato de servicio.
from software_fj.modelo import Sala, Equipo, Asesoria, DatoInvalido
# Cada servicio usa una tarifa ficticia en COP por hora.
servicios = [Sala('S1', 'Sala demo', 50000, 10),
             Equipo('E1', 'Portátiles demo', 10000, 2),
             Asesoria('A1', 'Asesoría demo', 80000, 'Python')]
# El mismo método produce resultados específicos sin condicionales por clase.
for servicio in servicios:
    print(servicio.describir(), '| Costo 2 horas:', servicio.calcular_costo(2))

Sala Sala demo; capacidad 10; COP 50000/hora | Costo 2 horas: 100000.00
Equipo Portátiles demo; 2 unidades; COP 10000/unidad/hora | Costo 2 horas: 40000.00
Asesoría Asesoría demo; Python; COP 80000/hora | Costo 2 horas: 160000.00


## 2. Variantes del cálculo y precisión

La variante con parámetros opcionales cumple la alternativa descrita en el
anexo: costo básico, con impuesto, con descuento o combinado. No se definen
múltiples métodos del mismo nombre, porque Python reemplazaría los anteriores.
La sobrescritura entre subclases es un concepto diferente.

El cálculo usa `Decimal`: `base × (1 − descuento) × (1 + impuesto)`.
Las tasas van de 0 a 1. El descuento se aplica antes del impuesto. El 19 % es
un ejemplo académico de cálculo, no una determinación tributaria.

In [2]:
# Se usa la misma sala para comparar cuatro variantes.
sala = servicios[0]
print('Base:', sala.calcular_costo(2))
print('Con impuesto:', sala.calcular_costo(2, impuesto='0.19'))
print('Con descuento:', sala.calcular_costo(2, descuento='0.10'))
print('Combinado:', sala.calcular_costo(2, impuesto='0.19', descuento='0.10'))

Base: 100000.00
Con impuesto: 119000.00
Con descuento: 90000.00
Combinado: 107100.00


## 3. Manejo de excepciones

Los errores del dominio heredan de `ErrorSoftwareFJ`. Las entidades validan
y lanzan errores; `ejecutar` es la frontera que los captura, registra y permite
continuar. Usa `try/except/else/finally`: `else` registra el éxito y `finally`
registra el fin de la operación, con independencia del resultado.

`numero` convierte entradas a `Decimal` y usa `raise DatoInvalido(...) from causa`
para conservar el error original. En la siguiente celda el registro incluye
ambas excepciones; la operación siguiente debe tener éxito.

In [3]:
# Se configura un archivo separado para esta demostración del encadenamiento.
from software_fj.ejecucion import configurar_log, ejecutar
logger = configurar_log('logs/notebook.log')
# La conversión de 'dos' genera un error con causa original preservada.
fallo = ejecutar('Conversión inválida', lambda: sala.calcular_costo('dos'), logger)
assert not fallo.correcto and isinstance(fallo.error, DatoInvalido)
assert fallo.error.__cause__ is not None
print('Error:', type(fallo.error).__name__, '| Causa:', type(fallo.error.__cause__).__name__)
# Una nueva operación confirma la continuidad real después del fallo.
posterior = ejecutar('Operación posterior', lambda: sala.calcular_costo(1), logger)
assert posterior.correcto
print('Operación posterior exitosa:', posterior.valor)

Error: DatoInvalido | Causa: InvalidOperation
Operación posterior exitosa: 50000.00


## 4. Simulación completa y resultados esperados

El lote crea un sistema nuevo y ejecuta 34 operaciones válidas e inválidas:
clientes, servicios, reservas, ocupación, cancelación, cálculos y errores de
entrada. Cada caso compara el resultado obtenido con el esperado. **Capturar
un error esperado cuenta como prueba correcta; no como operación exitosa.**

Los casos 20 y 21 muestran que un impuesto inválido no destruye la reserva:
el procesamiento posterior calcula COP 107100.00. Los casos finales confirman
que la aplicación continúa tras entradas no numéricas y no finitas.

In [4]:
# Se ejecuta la misma función que usa main.py, sin copiar la lógica del sistema.
from software_fj.simulaciones import simular
sistema, resultados = simular()
# Se comprueban los resultados y se cuenta cuántos errores eran esperados.
assert len(resultados) >= 10 and all(fila['cumple'] for fila in resultados)
errores_esperados = sum(fila['esperado'] != 'Éxito' for fila in resultados)
print('Casos con error esperado:', errores_esperados)
print('Clientes:', len(sistema.clientes), '| Servicios:', len(sistema.servicios))
# Se muestran los estados finales de las reservas sin datos de contacto.
for reserva in sistema.reservas:
    print(reserva.describir(), '| Total:', reserva.total)

OK | 01 Registrar cliente válido | Éxito
OK | 02 Rechazar correo inválido | DatoInvalido
OK | 03 Registrar cliente tras error | Éxito
OK | 04 Rechazar cliente duplicado | Duplicado
OK | 05 Rechazar parámetros faltantes | TypeError
OK | 06 Registrar sala | Éxito
OK | 07 Rechazar tarifa negativa | DatoInvalido
OK | 08 Registrar equipos | Éxito
OK | 09 Rechazar especialidad vacía | DatoInvalido
OK | 10 Registrar asesoría | Éxito
OK | 11 Rechazar capacidad inválida | DatoInvalido
OK | 12 Registrar servicio deshabilitado | Éxito
OK | 13 Rechazar servicio no disponible | ServicioNoDisponible
OK | 14 Rechazar cliente inexistente | NoEncontrado
OK | 15 Rechazar duración negativa | DatoInvalido
OK | 16 Crear reserva de sala | Éxito
OK | 17 Confirmar reserva de sala | Éxito
OK | 18 Crear segunda solicitud de sala | Éxito
OK | 19 Rechazar ocupación simultánea | ServicioNoDisponible
OK | 20 Rechazar impuesto inválido | DatoInvalido
OK | 21 Procesar sala tras error: COP 107100 | Éxito
OK | 22 Confi

## 5. Inspección del log

El log contiene inicio, resultado y fin de cada caso. Los errores incluyen
traceback. El archivo se abre en modo de anexar: repetir el notebook añade
otra ejecución. Para verificar un lote aislado puede utilizarse otra ruta al
llamar a `simular(ruta_log=...)`. Si el archivo no puede abrirse o escribirse,
se informa por consola; esa situación requiere corregir permisos o ruta.

En las salidas guardadas se omite la ruta absoluta del equipo por privacidad;
se conservan los nombres de archivos, las líneas y los mensajes del traceback.

In [5]:
# Se lee únicamente el archivo de eventos; no se persisten entidades del negocio.
contenido_log = Path('logs/software_fj.log').read_text(encoding='utf-8')
# La salida se limita a las últimas líneas; el archivo conserva el detalle completo.
print('\n'.join(contenido_log.splitlines()[-18:]))
assert 'ERROR_CONTROLADO' in contenido_log and 'ERROR_INESPERADO' in contenido_log
assert '34/34 casos cumplieron lo esperado' in contenido_log

  File "software_fj\ejecucion.py", line 60, in ejecutar
    valor = operacion()
  File "software_fj\simulaciones.py", line 78, in <lambda>
    caso('33 Rechazar descuento excesivo', lambda: sala.calcular_costo(1, descuento='1.01'), DatoInvalido)
                                                   ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "software_fj\modelo.py", line 181, in calcular_costo
    return self._total(self.tarifa * horas, impuesto, descuento)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "software_fj\modelo.py", line 164, in _total
    rebaja = numero(descuento, 'Descuento', maximo=Decimal('1'))
  File "software_fj\modelo.py", line 58, in numero
    raise DatoInvalido(f"{campo}: supera el límite {maximo}.")
software_fj.modelo.DatoInvalido: Descuento: supera el límite 1.
2026-11-22 14:14:53,544 | INFO | FIN | 33 Rechazar descuento excesivo
2026-11-22 14:14:53,544 | INFO | INICIO | 34 Operación válida final: COP 50000
2026-11-22 14:14:53,544 | IN

## 6. Conclusiones técnicas de la ejecución

- Los resultados muestran continuidad después de entradas inválidas y errores
  controlados, con eventos rastreables en el log.
- La separación entre entidades y frontera de ejecución mantiene las reglas
  del dominio independientes de la presentación de errores.
- El polimorfismo permite calcular costos de tres servicios a través del mismo
  método. La encapsulación protege identidad, contacto y estados.
- El proyecto no representa un sistema de producción: no incluye calendario,
  pagos, base de datos ni recuperación de los datos después de cerrar Python.

Estas conclusiones describen las celdas ejecutadas, no una garantía de puntaje.
Las pruebas adicionales se ejecutan con `python -m unittest discover -s tests -v`.
La colaboración de cinco integrantes solicitada en la rúbrica queda pendiente
de aclaración con el tutor, dado el desarrollo individual declarado.

## Materiales

Anexo 3, guía y rúbrica de Fase 4 de Programación 213023 (UNAD), suministrados
por el estudiante. El informe final incorporará las referencias APA de las
fuentes efectivamente revisadas. Ver `docs/MATRIZ_REQUISITOS.md`.